# Eksperimen IKMS 2026: K1, K2, gerbang mutu (K3), HHEM, BM25, dan lembar penilai
**Dijalankan oleh Tim A.** Menu *Runtime → Change runtime type → T4 GPU*. Jalankan sel berurutan.

| Tahap | Sel | Butuh | Perkiraan waktu |
|---|---|---|---|
| 1. Persiapan | 1–3 | berkas `ikms_paket.zip`, kunci API | 5 menit |
| 2. K2 (LLM) untuk 1.107 kejadian | 4–5 | kunci API | 15–40 menit |
| 3. **Lembar penilai + lembar τ** (kirim ke Tim B segera) | 6 | – | 1 menit |
| 4. HHEM dan AlignScore | 7–8 | GPU | 15–30 menit |
| 5. Tentukan τ, K3, retensi, BM25, fakta kunci | 9 | lembar τ yang sudah dilabeli Tim A | 2 menit |
| 6. Validasi audit | 10 | dua berkas Validasi_Audit dari Tim B | 1 menit |
| 7. Kappa + lembar diskusi | 11 | dua lembar penilaian utama dari Tim B | 1 menit |
| 8. Hasil akhir dan angka [A]–[J] | 12 | lembar diskusi yang sudah diputus | 1 menit |

Semua keluaran disimpan di Google Drive (`MyDrive/ikms2026`), sehingga aman bila Colab terputus. Sel 5 dapat dijalankan ulang dan akan melanjutkan dari kejadian terakhir.

In [1]:
# Sel 1 — Drive dan paket data
from google.colab import drive, files
drive.mount('/content/drive')
import os, zipfile
WORK = '/content/drive/MyDrive/ikms2026'
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f'{WORK}/ikms_lib.py'):
    print('Unggah ikms_paket.zip')
    up = files.upload()
    zipfile.ZipFile(list(up)[0]).extractall(WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))

Mounted at /content/drive
['KUNCI_validasi_audit_KHUSUS_TIM_A.csv', '__pycache__', 'data', 'ikms_lib.py', 'k2_cache.jsonl', 'run_meta.json']


In [2]:
# Sel 2 — Pustaka
!pip -q install openai anthropic rank_bm25 openpyxl
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, WORK)
import importlib, ikms_lib as L; importlib.reload(L)
ev = L.load_events('data/events.parquet')
ent = json.load(open('data/entity_dict.json'))
print(len(ev), 'kejadian |', ev['split'].value_counts().to_dict(), '|', ev['sample'].value_counts().to_dict())

1107 kejadian | {'dev': 635, 'test': 328, 'val': 144} | {'': 1047, 'tau15': 15, 'test40_other': 15, 'test40_ge6': 15, 'test40_rare': 10, 'calib5': 5}


In [3]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, os, time
env = dict(os.environ, OLLAMA_NUM_PARALLEL='4', OLLAMA_KEEP_ALIVE='-1', OLLAMA_CONTEXT_LENGTH='8192')
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT, env=env)
time.sleep(10)
!ollama pull qwen2.5:7b-instruct
!ollama list

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME                   ID              SIZE      MODIFIED               
qwen2.5:7b-instruct    845dbda0ea48    4.7 GB    Less than a second ago    


In [5]:
srv.terminate(); time.sleep(3)
env = dict(os.environ, OLLAMA_NUM_PARALLEL='4', OLLAMA_KEEP_ALIVE='-1', OLLAMA_CONTEXT_LENGTH='8192',
           LD_LIBRARY_PATH='/usr/lib64-nvidia')
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT, env=env)
time.sleep(10)
!grep -i -E "inference compute|gpu|cuda" /content/ollama.log | head -5

time=2026-11-13T08:12:20.086Z level=INFO source=routes.go:2474 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:8192 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:2562047h47m16.854775807s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:4 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:

In [7]:
from openai import OpenAI
import time
PROVIDER, MODEL, BASE_URL = 'ollama', 'qwen2.5:7b-instruct', 'http://localhost:11434/v1'
client = OpenAI(api_key='ollama', base_url=BASE_URL, timeout=600, max_retries=0)
PENALTI = {}   # kejadian yang butuh penalti pengulangan (dilaporkan di naskah)

def call(system, user):
    kw = dict(model=MODEL, temperature=0, max_tokens=1500, response_format={"type": "json_object"},
              messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    for fp in [0.0, 0.5, 1.0, 1.5]:
        try:
            r = client.chat.completions.create(frequency_penalty=fp, presence_penalty=fp / 2, **kw)
            if fp > 0: PENALTI[user[:80]] = fp
            return r.choices[0].message.content
        except Exception as ex:
            if 'repeat' not in str(ex): raise
    raise RuntimeError('model tetap mengulang setelah 4 percobaan')

for eid in ev[ev['sample'] == 'tau15'].event_id[:3]:
    t = time.time()
    s, u = L.build_messages(ev.at[eid, 'sentences'])
    try:
        cl_, kf_ = L.parse_k2(call(s, u), len(ev.at[eid, 'sentences']))
        print(f'--- kejadian {eid}: {time.time() - t:.1f} detik, {len(cl_)} klaim')
        print(pd.DataFrame(cl_)[['cid', 'text', 'sources']].to_string()[:1500])
    except Exception as ex:
        print(f'--- kejadian {eid}: GAGAL ({str(ex)[:120]})')
print('kejadian dengan penalti:', len(PENALTI))

--- kejadian 452: 9.9 detik, 6 klaim
         cid                                                                                                                                                         text sources
0  problem_1  Decision made to keep 36" hole opener assy at secure depth within template funnel and wait on WOW to minimise risk of damage to adjacent production risers.    [S2]
1    cause_1                                                                                                                                                      unknown      []
2  actions_1                                                                                                                                 Serviced drilling equipment.    [S4]
3  actions_2                                                                                        WOW with 36" hole opener assy at secure depth within template funnel.    [S3]
4   result_1                                               WOW: Wind Sea 

In [8]:
# Sel 4 — Catat metadata run (untuk naskah)
import datetime, platform
meta = {'provider': PROVIDER, 'model': MODEL, 'base_url': BASE_URL, 'temperature': 0,
        'tanggal_run': datetime.datetime.now().isoformat(timespec='minutes'), 'python': platform.python_version()}
json.dump(meta, open('run_meta.json', 'w'), indent=1); meta

{'provider': 'ollama',
 'model': 'qwen2.5:7b-instruct',
 'base_url': 'http://localhost:11434/v1',
 'temperature': 0,
 'tanggal_run': '2026-11-13T08:19',
 'python': '3.13.16'}

In [12]:
# Sel 5a — K2 untuk 60 kejadian prioritas (5 kalibrasi, 15 τ, 40 uji), dengan pemulihan otomatis
import collections, subprocess, time, os, json

def server_ok():
    try:
        client.chat.completions.create(model=MODEL, max_tokens=3, messages=[{"role": "user", "content": "OK"}])
        return True
    except Exception:
        return False

def nyalakan_server():
    subprocess.run(['pkill', '-f', 'ollama serve']); time.sleep(3)
    env = dict(os.environ, OLLAMA_NUM_PARALLEL='2', OLLAMA_KEEP_ALIVE='-1', OLLAMA_CONTEXT_LENGTH='8192',
               LD_LIBRARY_PATH='/usr/lib64-nvidia')
    global srv
    srv = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'),
                           stderr=subprocess.STDOUT, env=env)
    for _ in range(12):
        time.sleep(5)
        if server_ok():
            print('  server Ollama menyala'); return
    raise RuntimeError('server Ollama tidak mau menyala; kirim isi /content/ollama.log')

prio = set(ev[ev['sample'] != ''].event_id)
for putaran in range(1, 4):
    if not server_ok():
        print('Server mati, menyalakan ulang...'); nyalakan_server()
    k2_raw = L.run_k2(ev, call, 'k2_cache.jsonl', workers=2, only=prio)
    sisa = sorted(prio - set(k2_raw))
    print(f'Putaran {putaran}: prioritas selesai {60 - len(sisa)} / 60 | sisa {sisa}')
    if not sisa:
        break

gagal = [json.loads(l) for l in open('k2_cache.jsonl') if not json.loads(l).get('ok')]
print('Penyebab gagal terakhir:', collections.Counter(g['error'][:80] for g in gagal if g['event_id'] in sisa).most_common(3))
print('Kejadian dengan penalti:', len(PENALTI))

Sudah ada 32, akan diproses 28 kejadian
Selesai: 60 berhasil, 0 gagal []
Putaran 1: prioritas selesai 60 / 60 | sisa []
Penyebab gagal terakhir: []
Kejadian dengan penalti: 2


In [11]:
# Perbaikan: bersihkan sisa Gemini, cek log, mulai ulang Ollama dengan 2 permintaan paralel
import datetime
# 1) Simpan hanya hasil Qwen (setelah 15.10 WIB = 08.10 UTC); buang semua baris Gemini dan semua yang gagal
cut = datetime.datetime(2026, 11, 13, 8, 10, tzinfo=datetime.timezone.utc).timestamp()
lines = [json.loads(l) for l in open('k2_cache.jsonl')]
keep = [r for r in lines if r.get('ok') and r.get('ts', 0) >= cut]
os.rename('k2_cache.jsonl', 'k2_cache_campur_lama.jsonl')
with open('k2_cache.jsonl', 'w') as f:
    for r in keep: f.write(json.dumps(r) + '\n')
print('Hasil Qwen disimpan:', len(keep), '| prioritas:', len(prio & {r['event_id'] for r in keep}), '/ 60')

# 2) Lihat kenapa server mati
!tail -15 /content/ollama.log

# 3) Mulai ulang server dengan 2 permintaan paralel (lebih hemat memori GPU)
subprocess.run(['pkill', '-f', 'ollama serve']); time.sleep(3)
env = dict(os.environ, OLLAMA_NUM_PARALLEL='2', OLLAMA_KEEP_ALIVE='-1', OLLAMA_CONTEXT_LENGTH='8192',
           LD_LIBRARY_PATH='/usr/lib64-nvidia')
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT, env=env)
time.sleep(10)
print('Server hidup:', client.chat.completions.create(model=MODEL, max_tokens=5,
      messages=[{"role": "user", "content": "Say OK"}]).choices[0].message.content)

Hasil Qwen disimpan: 32 | prioritas: 32 / 60
slot   operator(): id  0 | task 3998 | new prompt, n_ctx_slot = 8192, n_keep = 4, task.n_tokens = 544
slot   operator(): id  0 | task 3998 | cached n_tokens = 84, memory_seq_rm [84, end)
slot init_sampler: id  0 | task 3998 | init sampler, took 0.10 ms, tokens: text = 544, total = 544
slot print_timing: id  2 | task 3910 | n_gen =    100, tg =  13.89 t/s, tg_3s =  14.03 t/s
slot print_timing: id  3 | task 3828 | n_gen =    192, tg =  15.06 t/s, tg_3s =  12.88 t/s
slot print_timing: id  2 | task 3910 | n_gen =    157, tg =  15.32 t/s, tg_3s =  18.67 t/s
slot print_timing: id  3 | task 3828 | n_gen =    248, tg =  15.75 t/s, tg_3s =  18.65 t/s
slot print_timing: id  1 | task 3982 | n_gen =    100, tg =  17.29 t/s, tg_3s =  17.46 t/s
slot print_timing: id  0 | task 3998 | n_gen =    100, tg =  18.66 t/s, tg_3s =  18.85 t/s
slot print_timing: id  2 | task 3910 | n_gen =    213, tg =  16.06 t/s, tg_3s =  18.55 t/s
slot print_timing: id  3 | task 

In [13]:
# Sel 6a — Lembar penilai (kirim ke Tim B) dan lembar τ (Tim A) dari 60 kejadian prioritas
k2_raw = {json.loads(l)['event_id']: json.loads(l)['raw'] for l in open('k2_cache.jsonl') if json.loads(l).get('ok')}
cl, kf = L.claims_table(ev, k2_raw)
cl = L.add_lexical_and_context(cl, ev, ent)
T = 'data/template_penilai.xlsx'
print('Kalibrasi:', L.make_rater_workbooks(ev, cl, ev[ev['sample'] == 'calib5'].event_id, T, 'Lembar_Penilai_Kalibrasi', 'Lembar penilaian KALIBRASI (5 kejadian validasi)', seed=21))
print('Utama    :', L.make_rater_workbooks(ev, cl, ev[ev['sample'].str.startswith('test40')].event_id, T, 'Lembar_Penilai_Utama', 'Lembar penilaian UTAMA (40 kejadian uji)', seed=22))
print('Label τ  :', L.make_tau_workbook(ev, cl, ev[ev['sample'] == 'tau15'].event_id, T, 'Lembar_Label_Tau', seed=11))
for f in ['Lembar_Penilai_Kalibrasi.xlsx', 'Lembar_Penilai_Utama.xlsx', 'Lembar_Label_Tau.xlsx']:
    files.download(f)

Kalibrasi: (45, 5)
Utama    : (413, 40)
Label τ  : (53, 15)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# Sel 5b — K2 untuk kejadian sisa (indeks penelusuran dan retensi)
k2_raw = L.run_k2(ev, call, 'k2_cache.jsonl', workers=4)

Sudah ada 60, akan diproses 1047 kejadian
  50/1047
  100/1047
  150/1047
  200/1047
  250/1047
  300/1047
  350/1047
  400/1047
  450/1047
  500/1047
  550/1047
  600/1047
  650/1047
  700/1047
  750/1047
  800/1047
  850/1047
  900/1047
  950/1047
  1000/1047
Selesai: 1107 berhasil, 0 gagal []


In [15]:
# Sel 6 — Tabel klaim lengkap + pemeriksaan leksikal (TANPA membuat ulang lembar penilai)
k2_raw = {json.loads(l)['event_id']: json.loads(l)['raw'] for l in open('k2_cache.jsonl') if json.loads(l).get('ok')}
hilang = sorted(set(ev.event_id) - set(k2_raw))
print(f'K2 tersedia: {len(k2_raw)} / {len(ev)} | tanpa K2: {len(hilang)}')
print('  per bagian data:', ev[ev.event_id.isin(hilang)]['split'].value_counts().to_dict())
cl, kf = L.claims_table(ev, k2_raw)
cl = L.add_lexical_and_context(cl, ev, ent)
cl.to_pickle('claims.pkl'); kf.to_pickle('key_facts.pkl')
print(cl.groupby(['cond', 'part']).size().unstack(0))
json.dump({'tanpa_k2': [int(e) for e in hilang], 'penalti': len(PENALTI) if 'PENALTI' in globals() else None},
          open('catatan_k2.json', 'w'))

K2 tersedia: 1107 / 1107 | tanpa K2: 0
  per bagian data: {}
cond         K1      K2
part                   
actions  1925.0  2404.0
cause       NaN  1104.0
lessons     NaN  1541.0
problem  1142.0  1223.0
result      NaN  1150.0


In [16]:
srv.terminate()

In [17]:
# Sel 7 — HHEM-2.1-Open (evaluator; terpisah dari gerbang mutu)
import torch
from transformers import AutoModelForSequenceClassification
hhem = AutoModelForSequenceClassification.from_pretrained('vectara/hallucination_evaluation_model', trust_remote_code=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    hhem = hhem.to(dev)
except Exception as ex:
    print('HHEM tetap di CPU:', ex)
cl = pd.read_pickle('claims.pkl')
mask = ~cl.unknown & (cl.context.str.len() > 0)
pairs = list(zip(cl.loc[mask, 'context'], cl.loc[mask, 'text']))
scores = []
for i in range(0, len(pairs), 32):
    scores += [float(x) for x in hhem.predict(pairs[i:i + 32])]
    if i % 1600 == 0: print(i, '/', len(pairs))
cl['hhem'] = np.nan
cl.loc[mask, 'hhem'] = scores
cl.to_pickle('claims.pkl'); print(cl.groupby('cond').hhem.describe())

config.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

configuration_hhem_v2.py:   0%|          | 0.00/760 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/vectara/hallucination_evaluation_model:
- configuration_hhem_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] You are using a model of type `HHEMv2Config` to instantiate a model of type `HHEMv2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


modeling_hhem_v2.py:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/vectara/hallucination_evaluation_model:
- modeling_hhem_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

... (7 outputs left out)

Loading weights:   0%|          | 0/113 [00:00<?, ?it/s]

0 / 8532


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1154 > 512). Running this sequence through the model will result in indexing errors


1600 / 8532
3200 / 8532
4800 / 8532
6400 / 8532
8000 / 8532
       count      mean       std       min       25%       50%       75%  \
cond                                                                       
K1    3067.0  0.912040  0.036463  0.658819  0.896062  0.918626  0.936996   
K2    5465.0  0.671629  0.354910  0.003513  0.294786  0.896898  0.939347   

           max  
cond            
K1    0.974333  
K2    0.995606  


In [19]:
# Sel 8 — AlignScore (skor gerbang mutu) untuk klaim fakta K2 — versi tambalan
import torch, transformers, nltk
transformers.AdamW = torch.optim.AdamW          # tambalan kompatibilitas (hanya dipakai saat pelatihan)
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
!wget -q -nc https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt
from alignscore import AlignScore
scorer = AlignScore(model='roberta-base', batch_size=32, device='cuda:0', ckpt_path='AlignScore-base.ckpt',
                    evaluation_mode='nli_sp')
cl = pd.read_pickle('claims.pkl')
m = (cl.cond == 'K2') & cl.part.isin(L.FACT_PARTS) & ~cl.unknown & (cl.context.str.len() > 0)
cl['align_score'] = np.nan
cl.loc[m, 'align_score'] = scorer.score(contexts=cl.loc[m, 'context'].tolist(), claims=cl.loc[m, 'text'].tolist())
cl.loc[(cl.cond == 'K2') & cl.part.isin(L.FACT_PARTS) & ~cl.unknown & (cl.context.str.len() == 0), 'align_score'] = 0.0
cl.to_pickle('claims.pkl'); print(cl.loc[m, 'align_score'].describe())

/usr/local/lib/python3.13/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (8 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Evaluating: 100%|██████████| 3948/3948 [02:50<00:00, 23.10it/s]

count    3948.000000
mean        0.875383
std         0.276940
min         0.000212
25%         0.953587
50%         0.992238
75%         0.999089
max         0.999355
Name: align_score, dtype: float64


In [21]:
# Sel 9 — Tentukan τ dari Lembar_Label_Tau, jalankan gerbang mutu (K3), retensi, BM25, fakta kunci
import json, numpy as np, pandas as pd
up = files.upload()                                   # pilih Lembar_Label_Tau.xlsx yang sudah diisi
cl = pd.read_pickle('claims.pkl'); kf = pd.read_pickle('key_facts.pkl')

# 1) Label Tim A + skor AlignScore
tl  = L.read_labels(list(up)[0])
key = pd.read_csv('Lembar_Label_Tau_KUNCI_TIM_A.csv')   # kunci dari Sel 6a
lab = (key.merge(tl, on='rid')
          .merge(cl[cl.cond == 'K2'][['event_id', 'cid', 'align_score', 'lex_ok']], on=['event_id', 'cid']))
lab = lab[lab.label.isin(['D', 'S', 'T']) & lab.align_score.notna()]
print('Klaim berlabel untuk τ:', len(lab), lab.label.value_counts().to_dict())

# 2) Pilih τ (F1 terbaik untuk mendeteksi klaim tidak didukung)
tau, grid = L.choose_tau(lab)
print(f'\nτ terpilih = {tau}')
print(grid[grid.tau == tau].round(3).to_string(index=False))
print('5 τ terbaik:\n', grid.sort_values('f1', ascending=False).head(5).round(3).to_string(index=False))

# 3) Gerbang mutu (K3) dan semua ukuran turunannya
cl = L.apply_gate(cl, tau)
cl.to_pickle('claims_gated.pkl')
T_rate, kept = L.retention(ev, cl)
p5  = L.bm25_p5(ev, cl)
kfa = L.key_fact_accuracy(ev, kf)
kept.to_pickle('retensi.pkl'); p5.to_pickle('p5.pkl'); kfa.to_pickle('fakta_kunci.pkl')
json.dump({'tau': float(tau), 'grid': grid.to_dict('records')}, open('tau.json', 'w'))

k2 = cl[cl.cond == 'K2']
print('\nK2 lolos gerbang (%):', round(100 * k2.passed.astype(float).mean(), 1))
print('  per bagian (%):', (k2.groupby('part').passed.apply(lambda s: 100 * s.astype(float).mean())).round(1).to_dict())
print('Retensi T (%):', round(100 * T_rate, 1), f'dari {len(kept)} kejadian uji layak pakai')
print('P@5 rerata:', p5.mean().round(3).to_dict(), f'({len(p5)} kueri)')
print('Akurasi fakta kunci (%):', (kfa[['jenis_masalah', 'kegagalan_peralatan', 'kedalaman']].mean() * 100).round(1).to_dict())

Saving Lembar_Label_Tau.xlsx to Lembar_Label_Tau (1).xlsx
Klaim berlabel untuk τ: 53 {'D': 45, 'T': 5, 'S': 3}

τ terpilih = 0.94
 tau  presisi  recall    f1
0.94      0.5   0.875 0.636
5 τ terbaik:
  tau  presisi  recall    f1
0.94    0.500   0.875 0.636
0.91    0.500   0.750 0.600
0.92    0.500   0.750 0.600
0.69    0.556   0.625 0.588
0.66    0.556   0.625 0.588

K2 lolos gerbang (%): 78.6
  per bagian (%): {'actions': 80.7, 'cause': 99.4, 'lessons': 59.0, 'problem': 66.3, 'result': 93.9}
Retensi T (%): 65.0 dari 246 kejadian uji layak pakai
P@5 rerata: {'K1': 0.42, 'K2': 0.48, 'K3': 0.373} (30 kueri)
Akurasi fakta kunci (%): {'jenis_masalah': 64.6, 'kegagalan_peralatan': 45.1, 'kedalaman': 85.7}


In [22]:
# Sel 10 — Validasi audit (dua berkas Validasi_Audit dari Tim B + kunci)
up = files.upload()            # unggah Audit_Penilai1.xlsx dan Audit_Penilai2.xlsx
p = sorted(up)
audit = L.audit_metrics(p[0], p[1], 'KUNCI_validasi_audit_KHUSUS_TIM_A.csv')
json.dump(audit, open('hasil_validasi_audit.json', 'w'), indent=1); audit
# Bila ada perbedaan, Tim B memutusnya lalu tulis CSV kolom ID,P1,P2,P3 (True/False) dan jalankan:
# L.audit_metrics(p[0], p[1], 'KUNCI_validasi_audit_KHUSUS_TIM_A.csv', resolved_csv='audit_putus.csv')

IndexError: list index out of range

In [23]:
# Sel 11 — Kappa penilaian utama + lembar diskusi (hanya klaim yang labelnya berbeda)
up = files.upload()            # unggah Penilai1_Nama.xlsx dan Penilai2_Nama.xlsx (lembar UTAMA)
p = sorted(up)
import shutil
for f in p: shutil.copy(f, f'ASLI_{f}')   # salinan label sebelum diskusi
kappa, merged = L.disagreement_sheet(p[0], p[1], 'Diskusi_Klaim_Berbeda.xlsx')
merged.to_pickle('label_dua_penilai.pkl'); json.dump({'kappa': kappa}, open('kappa.json', 'w'))
files.download('Diskusi_Klaim_Berbeda.xlsx')
# Tim B mengisi kolom label_akhir (D/S/T) bersama-sama, lalu jalankan Sel 12.

Saving Lembar_Penilai_Utama_Muhammad Reyvan Adrian.xlsx to Lembar_Penilai_Utama_Muhammad Reyvan Adrian.xlsx
Saving Lembar_Penilai_Utama_terisi_Muhammad Arzaq Kafabih.xlsx to Lembar_Penilai_Utama_terisi_Muhammad Arzaq Kafabih.xlsx
kappa biner = 0.633 | klaim dinilai keduanya = 413 | berbeda = 101


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# Sel 12 — Hasil akhir, Tabel 4, dan angka [A]–[J] untuk abstrak
up = files.upload()            # unggah Diskusi_Klaim_Berbeda.xlsx yang kolom label_akhir-nya sudah diisi
merged = pd.read_pickle('label_dua_penilai.pkl')
fin = L.final_labels(merged, list(up)[0])
cl = pd.read_pickle('claims_gated.pkl')
keymap = pd.read_csv('Lembar_Penilai_Utama_KUNCI_TIM_A.csv')
out, U = L.analyse(cl, fin, keymap, pd.read_pickle('retensi.pkl'), pd.read_pickle('p5.pkl'), pd.read_pickle('fakta_kunci.pkl'))
out['kappa'] = json.load(open('kappa.json'))['kappa']
out['tau'] = json.load(open('tau.json'))['tau']
json.dump(out, open('HASIL_AKHIR.json', 'w'), indent=1, default=float)
fmt = lambda x, d=1: f'{x:.{d}f}'.replace('.', ',')
w = out['wilcoxon_holm']['K2-K3']['p_holm']
isian = {'[A]': fmt(out['U_K2_%_pasangan']), '[B]': fmt(out['U_K3_%_pasangan']), '[C]': fmt(out['R_%']),
         '[D]': ('< 0,001' if w < 0.001 else '= ' + fmt(w, 3)), '[E]': fmt(out['T_%']),
         '[F]': fmt(out['spearman']['rho'], 2), '[G]': fmt(out['kappa'], 2),
         '[H]': fmt(out['P@5_rerata']['K2'], 2), '[I]': fmt(out['P@5_rerata']['K3'], 2),
         '[J]': ['nol', 'satu', 'dua', 'tiga', 'empat'][out['kriteria_tercapai']]}
print(json.dumps(out, indent=1, default=float)); print('\nISIAN ABSTRAK (Indonesia):', isian)
for f in ['HASIL_AKHIR.json', 'claims_gated.pkl', 'run_meta.json']:
    files.download(f)

Saving Diskusi_Klaim_Berbeda_final.xlsx to Diskusi_Klaim_Berbeda_final.xlsx
{
 "n_kejadian_lengkap_3_kondisi": 38,
 "U_rerata_%": {
  "K1": 13.9,
  "K2": 37.63,
  "K3": 30.54
 },
 "n_kejadian_per_kondisi": {
  "K1": 38,
  "K2": 40,
  "K3": 40
 },
 "friedman": {
  "chi2": 22.606,
  "p": 1.2335487303216621e-05,
  "n": 38
 },
 "wilcoxon_holm": {
  "K1-K2": {
   "p_holm": 2.715999256814406e-05,
   "r": 0.72,
   "n": 38
  },
  "K1-K3": {
   "p_holm": 0.007192370025814803,
   "r": 0.472,
   "n": 38
  },
  "K2-K3": {
   "p_holm": 0.05933346675499405,
   "r": 0.298,
   "n": 40
  }
 },
 "U_K2_%_pasangan": 37.63,
 "U_K3_%_pasangan": 30.54,
 "R_%": 18.82,
 "R_CI95_%": [
  3.73,
  35.71
 ],
 "T_%": 65.04,
 "T_CI95_%": [
  58.94,
  70.73
 ],
 "T_n_layak_uji": 246,
 "spearman": {
  "rho": 0.162,
  "p": 0.023231510166337094,
  "n_klaim": 196
 },
 "P@5_rerata": {
  "K1": 0.42,
  "K2": 0.48,
  "K3": 0.373
 },
 "P@5_friedman_p": 0.016851201259947823,
 "P@5_K3vsK2_p": 0.004066786325120292,
 "fakta_kunci_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# Sel 13 — ANALISIS SENSITIVITAS τ + ORACLE (eksploratif; hasil utama tetap τ = 0,94)
import numpy as np, pandas as pd, json
if 'ev' not in globals(): print('ev belum ada: jalankan dulu sel yang memuat data kejadian (Sel 2/6)')
cl0 = pd.read_pickle('claims.pkl')
if 'align' not in cl0: cl0['align'] = cl0['align_score']
if 'align_score' not in cl0: cl0['align_score'] = cl0['align']
fin = L.final_labels(pd.read_pickle('label_dua_penilai.pkl'), 'Diskusi_Klaim_Berbeda_final.xlsx')
keymap = pd.read_csv('Lembar_Penilai_Utama_KUNCI_TIM_A.csv')
kfa = pd.read_pickle('fakta_kunci.pkl')

rows = []
for t in [0.0, 0.5, 0.7, 0.8, 0.9, 0.94, 0.97, 0.99]:
    c = L.apply_gate(cl0, t)
    T, kept = L.retention(ev, c)
    p5 = L.bm25_p5(ev, c)
    o, _ = L.analyse(c, fin, keymap, kept, p5, kfa)
    rows.append({'tau': t, 'U_K3_%': o['U_rerata_%']['K3'], 'R_%': o['R_%'],
                 'p_holm_K2K3': round(o['wilcoxon_holm']['K2-K3']['p_holm'], 4),
                 'T_%': o['T_%'], 'P5_K3': o['P@5_rerata']['K3']})
sens = pd.DataFrame(rows); print(sens.to_string(index=False))

# Oracle: hanya klaim berlabel D yang lolos (40 kejadian uji berlabel)
lab = keymap.merge(fin, on='rid')
lab = lab[lab.cond == 'K2'].merge(cl0[cl0.cond == 'K2'][['event_id', 'cid', 'part', 'unknown']], on=['event_id', 'cid'])
ok = lab[(lab.label == 'D') & ~lab.unknown]
e40 = lab.event_id.unique()
orc = np.mean([((ok.event_id == e) & (ok.part == 'problem')).any() and ((ok.event_id == e) & (ok.part == 'actions')).any() for e in e40])
print('\nRetensi ORACLE pada 40 kejadian berlabel (%):', round(100 * orc, 1))
json.dump({'sensitivitas': rows, 'retensi_oracle_%': round(100 * orc, 1)}, open('sensitivitas.json', 'w'), indent=1, default=float)
files.download('sensitivitas.json')
print(json.load(open('catatan_k2.json')))

 tau  U_K3_%   R_%  p_holm_K2K3   T_%  P5_K3
0.00   35.51  5.62       0.0218 89.43  0.453
0.50   32.16 14.53       0.0249 78.86  0.440
0.70   32.29 14.19       0.0249 78.05  0.427
0.80   31.62 15.96       0.0250 76.42  0.413
0.90   30.17 19.82       0.0424 70.73  0.407
0.94   30.54 18.82       0.0593 65.04  0.373
0.97   29.36 21.98       0.0450 53.66  0.387
0.99   31.35 16.69       0.1648 36.59  0.373

Retensi ORACLE pada 40 kejadian berlabel (%): 45.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'tanpa_k2': [], 'penalti': 10}


In [27]:
# Sel 14 — kemas berkas (tidak mengubah apa pun)
import zipfile, os
daftar = ['events.parquet','entity_dict.json','catatan_k2.json','tau.json','retensi.pkl','p5.pkl',
          'fakta_kunci.pkl','key_facts.pkl','sensitivitas.json','ringkasan_sampel.json','kappa.json',
          'Lembar_Penilai_Utama_KUNCI_TIM_A.csv','Lembar_Label_Tau_KUNCI_TIM_A.csv']
with zipfile.ZipFile('paket_tahap1.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in daftar:
        if os.path.exists(f): z.write(f); print('masuk    :', f)
        else: print('TIDAK ADA:', f)
files.download('paket_tahap1.zip')

TIDAK ADA: events.parquet
TIDAK ADA: entity_dict.json
masuk    : catatan_k2.json
masuk    : tau.json
masuk    : retensi.pkl
masuk    : p5.pkl
masuk    : fakta_kunci.pkl
masuk    : key_facts.pkl
masuk    : sensitivitas.json
TIDAK ADA: ringkasan_sampel.json
masuk    : kappa.json
masuk    : Lembar_Penilai_Utama_KUNCI_TIM_A.csv
masuk    : Lembar_Label_Tau_KUNCI_TIM_A.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>